# Advanced Neural Operators

[Full course sequence](../../ai4sci/README.md) | **9/9 · Neural Operators** | Previous: [Climate](../climate/Multi-Physics_Climate_Modeling.ipynb)

Use PhysicsNeMo **2.2.2** to complete **Level 1 FNO → Level 2 AFNO → Level 3 PINO**.
All three levels use the same 2D reaction-diffusion problem and dataset splits.

| Level | File to edit | Configuration | Learning objective |
|---|---|---|---|
| 1 | [fno_physicsnemo_l1.py](fno_physicsnemo_l1.py) | [config_FNO.yaml](conf/config_FNO.yaml) | Create separate TensorDataset splits and an FNO model |
| 2 | [fno_physicsnemo_l2.py](fno_physicsnemo_l2.py) | [config_AFNO.yaml](conf/config_AFNO.yaml) | Understand the relationship between AFNO patch size and grid size |
| 3 | [fno_physicsnemo_l3.py](fno_physicsnemo_l3.py) | [config_PINO.yaml](conf/config_PINO.yaml) | Define the symbolic PDE residual and train PINO |

In JupyterLab, **edit the `FIXME` functions in the actual `.py` file → save the file → run the corresponding notebook cell**.
Editing a Markdown example does not change the executable program. The training loop is in
[operator_training.py](operator_training.py), where you can follow `optimizer.zero_grad → forward → loss.backward → optimizer.step` directly.

Participant mode is the default. If an exercise is incomplete, the program identifies the function to complete and exits.
Instructors can set `REFERENCE = True` below or add `--reference` on the command line to run the reference implementation.
This option selects the completed exercise functions; training still runs from initialization rather than loading pretrained results.

## Basic Fourier Neural Operator (FNO)

Neural operators learn mappings between function spaces. Here the input is an entire forcing field and the output is its solution field, not a single point value.

### Mathematical formulation

$$u(x,y)-\Delta u(x,y)=f(x,y),\qquad (x,y)\in[0,1]^2$$

with periodic boundary conditions. The learned operator is $\mathcal G:f\mapsto u=(1-\Delta)^{-1}f$.
The discrete grid contains **$x_i=i/N$, $y_j=j/N$, $i,j=0,\ldots,N-1$**, excluding the duplicate periodic endpoint at 1.
This gives spacing $1/N$ and matches spectral differentiation.

The input and output do **not** generally have similar scales: high-frequency forcing coefficients are divided by
$1+(2\pi)^2(k^2+l^2)$. Compute separate input/output means and standard deviations from **training data only**, use the same values for validation and test data, and restore physical units before computing the PDE residual.

This exercise uses synthetic reaction-diffusion fields. Darcy flow, weather prediction and magnetohydrodynamics are applications of neural operators, but are not the PDE or dataset used by these three scripts.

## Execution settings

The cells below run each program in the notebook kernel's Python environment and stop if the program fails.
`STEPS = 200` provides a starting run length and does not guarantee convergence. Small experiments can run on a CPU.
Set `DEVICE = "cuda"` to require a GPU. Each level writes its results to a separate directory.

The configuration files set `training.steps` to 10,000; `--steps` overrides this value. These programs use the explicit PyTorch training API instead of the legacy Sym `Domain`, `Solver`, `Key`, and `*Arch` interfaces.

Instructors and automated checks can run the same cells using the environment variables
`AI4SCI_DEVICE`, `AI4SCI_STEPS`, `AI4SCI_REFERENCE=1`, `AI4SCI_OUTPUT_DIR`, and `AI4SCI_DATA_DIR`.
Each execution cell displays the actual Python command. The default output directory receives a new name for each run.
When changing the dataset path, retain a **64×64** grid to match the default lesson configurations.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
from datetime import datetime
from uuid import uuid4
from IPython.display import Image, display

# Locate this lesson from its own directory or from the repository root.
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / "challenge" / "neural_operator")
    match = next((path for path in candidates if (path / "Advanced_Neural_Operators.ipynb").exists()), None)
    if match is not None:
        LESSON_DIR = match.resolve()
        break
else:
    raise FileNotFoundError("Open the notebook inside the cloned bootcamp repository.")

DEVICE = os.environ.get("AI4SCI_DEVICE", "auto")  # also accepts "cpu" or "cuda"
STEPS = int(os.environ.get("AI4SCI_STEPS", "200"))
SEED = 42
REFERENCE = os.environ.get("AI4SCI_REFERENCE", "0").lower() in {"1", "true", "yes"}
DATA_DIR = Path(os.environ.get("AI4SCI_DATA_DIR", str(LESSON_DIR / "datasets" / "Reaction_Diffusion"))).expanduser().resolve()
run_name = datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:8]
RUN_DIR = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LESSON_DIR / "outputs" / run_name))).expanduser().resolve()

def run_script(script, *arguments):
    command = [sys.executable, str(LESSON_DIR / script), *map(str, arguments)]
    subprocess.run(command, cwd=LESSON_DIR, check=True)

def inspect_level(level):
    output = RUN_DIR / f"level{level}"
    metrics = json.loads((output / "metrics.json").read_text())
    print(json.dumps({"initial_test": metrics["initial_test"], "final_test": metrics["test"],
                      "convergence_claim": metrics["convergence_claim"]}, indent=2))
    if (output / "preview.png").exists():
        display(Image(filename=str(output / "preview.png")))
    return metrics

print({"python": sys.executable, "lesson": str(LESSON_DIR), "device": DEVICE})

## Step 0: Data generation

[generate_data.py](generate_data.py) constructs forcing fields from random sine/cosine tensor-product Fourier modes up to $K=6$.
For each coefficient $a_{kl}$ multiplying a basis function $\phi_{kl}$:

$$f=\sum_{k,l} a_{kl}\phi_{kl},\qquad
u=\sum_{k,l}\frac{a_{kl}}{1+(2\pi)^2(k^2+l^2)}\phi_{kl}.$$

The generator uses `einsum("bkl,kx,ly->bxy", ...)` so both spatial axes are retained correctly.
All modes lie below the Nyquist frequency. An independent FFT residual checks the analytical pairs before float32 storage.

Default files, with shape `[samples, 1, 64, 64]`:

| Split | Samples | Role |
|---|---:|---|
| `train.hdf5` | 8,000 | Gradient updates and normalization statistics |
| `val.hdf5` | 1,000 | Development evaluation |
| `test.hdf5` | 1,000 | Held-out comparison; never used by the optimizer |

The three random streams are spawned independently from `--seed`. The generator writes a schema and split identity to HDF5 plus `manifest.json`.
The historical `datasets/Poisson_Fourier` files remain in the repository for provenance and are **not** training data for this PDE. The loader rejects their old schema.

```bash
python generate_data.py --seed 42
```

Existing dataset files are preserved; regeneration requires a fresh `--output-dir` and the corresponding training `--data-dir`.
For a CPU smoke check, use `--train-samples 16 --val-samples 4 --test-samples 4` with the same 64×64 grid.
Reducing the grid also requires matching `data.grid_size`, FNO modes and AFNO patch sizes in a separate configuration.

In [ ]:
# Generate once, then reuse the same independent splits for all three levels.
required_files = [DATA_DIR / f"{split}.hdf5" for split in ("train", "val", "test")]
if all(path.exists() for path in required_files):
    sys.path.insert(0, str(LESSON_DIR))
    from operator_training import load_data
    load_data(DATA_DIR)
    print("Reusing validated reaction-diffusion dataset:", DATA_DIR)
else:
    run_script("generate_data.py", "--seed", SEED, "--output-dir", DATA_DIR)

## Level 1 Fourier Neural Operator

FNO captures global correlations by Fourier transformation, learned multiplication of retained modes, inverse transformation, and a local skip/activation path:

$$v_{j+1}=\sigma\left(W_jv_j+\mathcal F^{-1}\big(R_j\cdot\mathcal F(v_j)\big)\right).$$

The FFT contributes $O(N\log N)$ work; total model cost also depends on channels, layers and retained modes.
`FNO` is a tensor-in/tensor-out PyTorch module. It does not take a dictionary of `Key` objects.

```python
from torch.utils.data import TensorDataset
from physicsnemo.models.fno import FNO

train_dataset = TensorDataset(normalized_f_train, normalized_u_train)
model = FNO(in_channels=1, out_channels=1, dimension=2,
            latent_channels=32, num_fno_layers=4, num_fno_modes=12,
            padding=0, coord_features=False)
prediction = model(normalized_f_batch)  # [batch, 1, N, N]
loss = (prediction - normalized_u_batch).square().mean()
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()
```

**Exercise:** implement `build_datasets` and `build_model` in [fno_physicsnemo_l1.py](fno_physicsnemo_l1.py), then save.
Use three separate `TensorDataset` objects. The output shape must equal the input shape.
Periodic data use zero padding width and no nonperiodic coordinate channels in this baseline.
The trainer performs normalization outside the model and restores physical units for evaluation.

In [ ]:
# Terminal equivalent from challenge/neural_operator:
# python fno_physicsnemo_l1.py --device cpu --steps 200 --output-dir outputs/my-l1
# Add --reference for the instructor implementation.
command = [sys.executable, str(LESSON_DIR / "fno_physicsnemo_l1.py"),
           "--device", DEVICE, "--steps", str(STEPS), "--seed", str(SEED),
           "--data-dir", str(DATA_DIR), "--output-dir", str(RUN_DIR / "level1")]
if REFERENCE:
    command.append("--reference")
print(command)
subprocess.run(command, cwd=LESSON_DIR, check=True)

In [ ]:
metrics_l1 = inspect_level(1)

## Level 2 Adaptive Fourier Neural Operator

AFNO uses patch embedding, FFT over the patch grid, block-diagonal channel mixing with nonlinearities and sparsity, and reconstruction to the output grid.
FNO's learned spectral matrices are also trained; calling them “fixed weights” does not mean they are untrained. AFNO provides a different token-mixing architecture, not a guaranteed accuracy improvement.

```python
from physicsnemo.models.afno import AFNO
model = AFNO(inp_shape=[64, 64], in_channels=1, out_channels=1,
             patch_size=[4, 4], embed_dim=64, depth=4, num_blocks=8)
prediction = model(normalized_f_batch)
```

**Exercise:** implement the same separate datasets and instantiate `AFNO` in [fno_physicsnemo_l2.py](fno_physicsnemo_l2.py).
Check that both image dimensions are divisible by the patch dimensions. Do not crop the periodic square: cropping would change the domain and the PDE boundary conditions.
`embed_dim` must also be divisible by `num_blocks`.

Use the same training/validation/test split as Level 1. Compare held-out relative L2 error, not just training loss. AFNO is used in weather applications, but this experiment remains the same reaction-diffusion operator.

In [ ]:
# Terminal equivalent from challenge/neural_operator:
# python fno_physicsnemo_l2.py --device cpu --steps 200 --output-dir outputs/my-l2
# Add --reference for the instructor implementation.
command = [sys.executable, str(LESSON_DIR / "fno_physicsnemo_l2.py"),
           "--device", DEVICE, "--steps", str(STEPS), "--seed", str(SEED),
           "--data-dir", str(DATA_DIR), "--output-dir", str(RUN_DIR / "level2")]
if REFERENCE:
    command.append("--reference")
print(command)
subprocess.run(command, cwd=LESSON_DIR, check=True)

In [ ]:
metrics_l2 = inspect_level(2)

## Level 3 Physics-Informed Neural Operator

PINO combines a neural-operator backbone with data and physics losses. Here the backbone is the same FNO as Level 1:

$$\mathcal L=\mathcal L_{\rm data}+\lambda\mathcal L_{\rm physics},\qquad
r=u_{\rm pred}-\Delta u_{\rm pred}-f.$$

The implementation compares normalized solutions for data loss, and uses $\operatorname{mean}[(r/\sigma_f)^2]$ for physics loss, where $\sigma_f$ is the **training** forcing standard deviation. This controls scales without changing the PDE. Residual metrics are reported again in physical units.

`physicsnemo.sym` is the symbolic PDE component inside the unified package:

```python
from sympy import Function, Symbol
from physicsnemo.sym.eq.pde import PDE
from physicsnemo.sym.eq.phy_informer import PhysicsInformer
from operator_training import BatchedPhysicsInformer

class ReactionDiffusion(PDE):
    def __init__(self):
        self.dim = 2
        x, y = Symbol("x"), Symbol("y")
        u, f = Function("u")(x, y), Function("f")(x, y)
        self.equations = {"reaction_diffusion": u - u.diff(x, 2) - u.diff(y, 2) - f}

physics = BatchedPhysicsInformer(PhysicsInformer(
    required_outputs=["reaction_diffusion"], equations=ReactionDiffusion(),
    grad_method="spectral", bounds=[1.0, 1.0], device="cpu"))
residual = physics.forward({"u": predicted_u_physical, "f": forcing_physical})["reaction_diffusion"]
physics_loss = (residual / training_f_std).square().mean()
```

**Exercise:** complete the dataset, FNO and `ReactionDiffusionPDE.equations` functions in [fno_physicsnemo_l3.py](fno_physicsnemo_l3.py).
The provided `build_physics` connects this equation to spectral differentiation. The explicit training loop backpropagates through both losses into FNO.

For the periodic unit square, spectral second derivatives multiply Fourier coefficients by $-(2\pi k)^2$ and $-(2\pi l)^2$.
The provided `BatchedPhysicsInformer` evaluates each sample separately because the upstream spectral adapter operates on one scalar field. This preserves batch independence and gradients.
The evaluator separately computes the full Laplacian with `torch.fft.fft2` and checks it agrees with `PhysicsInformer`.
Adding a physics loss does not guarantee physical consistency or better generalization; both must be measured on held-out fields.

In [ ]:
# Terminal equivalent from challenge/neural_operator:
# python fno_physicsnemo_l3.py --device cpu --steps 200 --output-dir outputs/my-l3
# Add --reference for the instructor implementation.
command = [sys.executable, str(LESSON_DIR / "fno_physicsnemo_l3.py"),
           "--device", DEVICE, "--steps", str(STEPS), "--seed", str(SEED),
           "--data-dir", str(DATA_DIR), "--output-dir", str(RUN_DIR / "level3")]
if REFERENCE:
    command.append("--reference")
print(command)
subprocess.run(command, cwd=LESSON_DIR, check=True)

In [ ]:
metrics_l3 = inspect_level(3)

## Summary and comparison

| Aspect | FNO | AFNO | PINO in this lesson |
|---|---|---|---|
| Model | FNO spectral layers | AFNO patch/token mixing | Same FNO as Level 1 |
| Training objective | Data loss | Data loss | Data loss + PDE residual |
| PDE residual in training | No | No | Unified `physicsnemo.sym` |
| Validation | Held-out solution and residual metrics | Same | Same + independent residual agreement |

FNO is a useful operator-learning baseline. AFNO provides a patch-based architecture that is also used for large image-like scientific fields. PINO is useful when a governing equation is available and one wants to test whether physics information helps. None of these choices guarantees better accuracy, lower data requirements or reliable extrapolation by itself.

Every successful run writes `metrics.json`, `loss.csv`, `model.pt`, `predictions.npz`, and `preview.png` when matplotlib is available. The checkpoint includes the model settings and training-only normalization statistics.

Compare `test_relative_l2_before` and `test_relative_l2_after`, the physical-unit PDE RMSE, and the spatial error plot across Levels. Loss values from different objectives are not directly comparable. A two-step smoke test checks execution and finite gradients; it is not evidence of convergence or teaching-run quality.

In [ ]:
for level in (1, 2, 3):
    path = RUN_DIR / f"level{level}" / "metrics.json"
    if path.exists():
        result = json.loads(path.read_text())
        print(result["method"], {
            "steps": result["steps"],
            "relative_l2_before": result["test_relative_l2_before"],
            "relative_l2_after": result["test_relative_l2_after"],
            "pde_rmse_fft": result["test"]["pde_rmse_fft"],
        })

## References and further reading

- [FNO: Li et al., Fourier Neural Operator for Parametric PDEs](https://arxiv.org/abs/2010.08895)
- [AFNO: Guibas et al., Adaptive Fourier Neural Operators](https://arxiv.org/abs/2111.13587)
- [PINO: Li et al., Physics-Informed Neural Operator](https://arxiv.org/abs/2111.03794)
- [Unified PhysicsNeMo repository](https://github.com/NVIDIA/physicsnemo)
- [Official FNO and AFNO API](https://docs.nvidia.com/physicsnemo/latest/physicsnemo/api/models/fnos.html)
- [Official PDE and PhysicsInformer API](https://docs.nvidia.com/physicsnemo/latest/physicsnemo/api/physicsnemo.sym.html)
- [Official example catalog, including Darcy flow](https://docs.nvidia.com/physicsnemo/latest/physicsnemo/examples/index.html)

### Next steps

[Full course sequence](../../ai4sci/README.md) | **9/9 · Neural Operators** | Previous: [Climate](../climate/Multi-Physics_Climate_Modeling.ipynb)

Summarize how the three model/training approaches compare on the same data, then discuss prediction error, runtime, and PDE residuals with the instructor.

--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.
